# Logistic Regression - GA Optimization

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import time
from data_loader import load_clinc
from ga_optimizer import GeneticOptimizer

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = True
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda
1525 / 620 / 1100, 151 classes (sample)


In [2]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [3]:
PATIENCE = 5
MAX_EPOCHS = 50


def evaluate_hyperparams(genes):
    torch.manual_seed(42)
    np.random.seed(42)

    vectorizer = TfidfVectorizer(
        max_features=genes['max_features'],
        ngram_range=(1, genes['ngram_max']),
        min_df=2, max_df=0.95
    )
    tr_feat = vectorizer.fit_transform(train_texts).toarray()
    vl_feat = vectorizer.transform(val_texts).toarray()

    tr_loader = DataLoader(TextDataset(tr_feat, train_labels), batch_size=genes['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(vl_feat, val_labels), batch_size=genes['batch_size'], shuffle=False)

    model = LogisticRegression(tr_feat.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=genes['learning_rate'], weight_decay=genes['weight_decay'])

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                all_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    return best_f1, {"val_f1": round(best_f1, 4), "epochs": epoch + 1}

In [4]:
search_space = {
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'max_features': [1000, 2000, 3000, 5000, 8000, 10000],
    'ngram_max': [1, 2, 3],
    'batch_size': [32, 64, 128, 256],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3]
}

ga = GeneticOptimizer(
    search_space=search_space,
    fitness_fn=evaluate_hyperparams,
    population_size=20,
    n_generations=15,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_size=3,
    elitism_count=2,
    seed=42
)

ga_results = ga.run()
print(f"\nBest F1: {ga_results['best_fitness']:.4f}")
print(f"Best params: {ga_results['best_genes']}")
print(f"Total evaluations: {ga_results['total_evaluations']}")
print(f"Total time: {ga_results['total_time_seconds']:.1f}s")

Gen  1/15  best=0.7053  avg=0.5528  global_best=0.7053
Gen  2/15  best=0.7081  avg=0.6272  global_best=0.7081
Gen  3/15  best=0.7081  avg=0.6532  global_best=0.7081
Gen  4/15  best=0.7081  avg=0.6812  global_best=0.7081
Gen  5/15  best=0.7081  avg=0.6764  global_best=0.7081
Gen  6/15  best=0.7081  avg=0.6862  global_best=0.7081
Gen  7/15  best=0.7081  avg=0.6817  global_best=0.7081
Gen  8/15  best=0.7081  avg=0.6589  global_best=0.7081
Gen  9/15  best=0.7081  avg=0.6601  global_best=0.7081
Gen 10/15  best=0.7081  avg=0.6945  global_best=0.7081
Gen 11/15  best=0.7081  avg=0.6866  global_best=0.7081
Gen 12/15  best=0.7081  avg=0.6766  global_best=0.7081
Gen 13/15  best=0.7081  avg=0.6772  global_best=0.7081
Gen 14/15  best=0.7081  avg=0.6699  global_best=0.7081
Gen 15/15  best=0.7081  avg=0.6677  global_best=0.7081

Best F1: 0.7081
Best params: {'learning_rate': 0.05, 'max_features': 1000, 'ngram_max': 1, 'batch_size': 128, 'weight_decay': 0.0001}
Total evaluations: 290
Total time: 458.3

In [5]:
best = ga_results['best_genes']
torch.manual_seed(42)
np.random.seed(42)

vectorizer = TfidfVectorizer(
    max_features=best['max_features'],
    ngram_range=(1, best['ngram_max']),
    min_df=2, max_df=0.95
)
tr_feat = vectorizer.fit_transform(train_texts).toarray()
vl_feat = vectorizer.transform(val_texts).toarray()
te_feat = vectorizer.transform(test_texts).toarray()

tr_loader = DataLoader(TextDataset(tr_feat, train_labels), batch_size=best['batch_size'], shuffle=True)
vl_loader = DataLoader(TextDataset(vl_feat, val_labels), batch_size=best['batch_size'], shuffle=False)
te_loader = DataLoader(TextDataset(te_feat, test_labels), batch_size=best['batch_size'], shuffle=False)

model = LogisticRegression(tr_feat.shape[1], num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best['learning_rate'], weight_decay=best['weight_decay'])

best_f1, best_state, patience_counter = 0, None, 0

for epoch in range(MAX_EPOCHS):
    model.train()
    for features, labels in tr_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    vl_preds = []
    with torch.no_grad():
        for features, labels in vl_loader:
            vl_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

    vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
    if vl_f1 > best_f1:
        best_f1, best_state, patience_counter = vl_f1, model.state_dict().copy(), 0
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        break

model.load_state_dict(best_state)
model.eval()
te_preds = []
with torch.no_grad():
    for features, labels in te_loader:
        te_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

acc = accuracy_score(test_labels, te_preds)
p = precision_score(test_labels, te_preds, average='macro', zero_division=0)
r = recall_score(test_labels, te_preds, average='macro', zero_division=0)
f1_m = f1_score(test_labels, te_preds, average='macro', zero_division=0)
f1_w = f1_score(test_labels, te_preds, average='weighted', zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1 macro:   {f1_m:.4f}")
print(f"F1 weighted:{f1_w:.4f}")

Accuracy:   0.6136
Precision:  0.6381
Recall:     0.6840
F1 macro:   0.6356
F1 weighted:0.5926


In [6]:
results = {
    "model_type": "LogisticRegression",
    "optimization": "genetic_algorithm",
    "best_hyperparameters": ga_results['best_genes'],
    "total_evaluations": ga_results['total_evaluations'],
    "search_time_seconds": ga_results['total_time_seconds'],
    "test_metrics": {
        "accuracy": round(acc, 4),
        "macro_precision": round(p, 4),
        "macro_recall": round(r, 4),
        "macro_f1": round(f1_m, 4),
        "weighted_f1": round(f1_w, 4)
    },
    "ga_params": {
        "population_size": 20,
        "n_generations": 15,
        "crossover_rate": 0.8,
        "mutation_rate": 0.2,
        "tournament_size": 3,
        "elitism_count": 2
    }
}

with open('results/results_lr_ga.json', 'w') as f:
    json.dump(results, f, indent=4, default=str)

ga.save_results('results/results_lr_ga_full.json')

print("Saved: results/results_lr_ga.json, results/results_lr_ga_full.json")

Saved: results/results_lr_ga.json, results/results_lr_ga_full.json
